# Implementation of Poisson equation in 2D (Cartesian)

In [3]:
import logging
import numpy as np
from matplotlib import pyplot as plt
import multigrid2d_numba as multi
from scipy.special import erf
from timeit import default_timer
from numba import njit

In [4]:
@njit
def DirichletDirichlet(_gridx, _gridy, y, level):
    y[0,:] = 0
    y[-1,:] = 0
    y[:,0] = 0
    y[:,-1] = 0


In [5]:
def setup_grid(num_xcells, num_ycells, x_start, x_end, y_start, y_end):

    gridx = np.linspace(x_start, x_end, num_xcells + 1)
    gridy = np.linspace(y_start, y_end, num_ycells + 1)

    gridy, gridx = np.meshgrid(gridy, gridx)

    hx  = gridx[1,0] - gridx[0,0]
    hy  = gridy[1,0] - gridy[0,0]
    src = np.sin(gridy)*np.cos(gridx)
    mg = multi.Multigrid(
        num_xcells=num_xcells,
        num_ycells=num_ycells,
        gridx=gridx,
        gridy=gridy,
        src=src,
        omega_sor=1.8,
        omega_high_error_limit=1,
        omega_low_error_limit=2e-8,
        boundary=DirichletDirichlet,
        min_cells=20,
        tol=2.0e-9,
        red_black_decouple=True,
        direct_restriction=False,
    )

    mg.logger.handlers[0].setLevel(logging.DEBUG)
    mg.logger.setLevel(logging.DEBUG)
    return mg, gridx, gridy

In [6]:
mg, gridx, gridy = setup_grid(1024, 1024, 0.0, 10.0, 0.0,10.0)
start = default_timer()
mg.solve_to_tolerance(smoothing_iters=100, max_vcycle_iters=100, subcycles=2)
end = default_timer()
print(f"solve took {end - start} seconds")
print(f"solve required {mg._iterations_required} iterations on parent grid")

err=np.float64(0.9999984694881129) at it=0
Starting Vcycle
 restrict at 1024
 *  restrict at 512
 *  *  restrict at 256
 *  *  *  restrict at 128
 *  *  *  *  restrict at 64
 *  *  *  *  *  prolongate at 32
 *  *  *  *  restrict at 64
 *  *  *  *  *  prolongate at 32
 *  *  *  *  prolongate at 64
 *  *  *  restrict at 128
 *  *  *  *  restrict at 64
 *  *  *  *  *  prolongate at 32
 *  *  *  *  prolongate at 64
 *  *  *  prolongate at 128
 *  *  restrict at 256
 *  *  *  restrict at 128
 *  *  *  *  restrict at 64
 *  *  *  *  *  prolongate at 32
 *  *  *  *  prolongate at 64
 *  *  *  restrict at 128
 *  *  *  *  prolongate at 64
 *  *  *  prolongate at 128
 *  *  prolongate at 256
 *  restrict at 512
 *  *  restrict at 256
 *  *  *  restrict at 128
 *  *  *  *  restrict at 64
 *  *  *  *  *  prolongate at 32
 *  *  *  *  prolongate at 64
 *  *  *  restrict at 128
 *  *  *  *  prolongate at 64
 *  *  *  prolongate at 128
 *  *  restrict at 256
 *  *  *  restrict at 128
 *  *  *  *  pr

solve took 6.1099776229966665 seconds
solve required 799 iterations on parent grid


In [5]:
def iterate_to_solution(mg, n_iters=1000, tol=2.0e-9):
    start = default_timer() 
    mg.calculate_defect()
    err = mg.defect_linf
    print(err)
    while err > tol:
        mg.gauss_seidel_smoothing(n_iters=n_iters, omega_override=1.8)
        mg.calculate_defect()
        err = mg.defect_linf
        end = default_timer()
        print(f"solve taking at least {end - start} seconds")
        print(err)
    end = default_timer()
    print(f"solve took  {end - start} seconds")
    print(f"solve required {mg._iterations_required} iterations on parent grid") 

In [6]:
mg, gridx, gridy = setup_grid(1024, 1024, 0.0, 10.0, 0.0,10.0)
iterate_to_solution(mg, n_iters=1000, tol=1.0e-8)

0.9999984694881129
solve taking at least 3.3945175220001147 seconds
4.321760401141551
solve taking at least 6.796205205000206 seconds
2.0726718981705554
solve taking at least 10.207853605999844 seconds
1.1012810008402856
solve taking at least 13.580136163000134 seconds
0.6304109308138803
solve taking at least 16.970828441999856 seconds
0.3782354277806199
solve taking at least 20.363258038999902 seconds
0.23360920179317057
solve taking at least 23.75126205100014 seconds
0.1475892330329428
solve taking at least 27.145497114000136 seconds
0.09535526893448831
solve taking at least 30.525227573999928 seconds
0.06309820190683046
solve taking at least 33.91278766100004 seconds
0.042827253626464
solve taking at least 37.315817545999835 seconds
0.02983713601732063
solve taking at least 40.697457192 seconds
0.02133223810939866
solve taking at least 44.083019783000054 seconds
0.01563266558420151
solve taking at least 47.47450482000022 seconds
0.011718432791825495
solve taking at least 50.86989149